# Difficulty classifier fine-tuning

Runs entirely in Colab. Do **not** run this locally -- it installs torch/transformers/datasets, which is exactly the heavy ML footprint the rest of this project's local codebase deliberately avoids.

Steps: 1) install deps, 2) upload `prepare_dataset.py` + `train_classifier.py` from the repo's `training/` folder, 3) build the dataset, 4) fine-tune, 5) zip the checkpoint and download it, 6) copy that folder back into the repo at `training/checkpoints/difficulty-classifier/` (or wherever `classifier_checkpoint_dir` in `app/config.py` points).

Runtime -> Change runtime type -> GPU, before running.

In [ ]:
!pip install -q torch transformers datasets scikit-learn accelerate

## Upload the training scripts

Upload `prepare_dataset.py` and `train_classifier.py` from this repo's `training/` folder (Files pane on the left -> upload, or run the cell below to use Colab's upload widget).

In [ ]:
from google.colab import files

uploaded = files.upload()  # select prepare_dataset.py and train_classifier.py
assert "prepare_dataset.py" in uploaded and "train_classifier.py" in uploaded, (
    "Both prepare_dataset.py and train_classifier.py must be uploaded before continuing."
)

In [ ]:
import os

os.makedirs("training", exist_ok=True)
os.rename("prepare_dataset.py", "training/prepare_dataset.py")
os.rename("train_classifier.py", "training/train_classifier.py")

## Build the dataset

Downloads MMLU + GSM8K via `datasets` and writes the labeled JSONL. See the docstring in `prepare_dataset.py` for the labeling strategy and its known limitations -- it's a defensible placeholder, not ground truth.

In [ ]:
!python training/prepare_dataset.py --per-class 500 --output training/data/difficulty_dataset.jsonl

## Fine-tune

Default base model is MiniLM (small, trains quickly on a free Colab GPU). Swap `--base-model` for `distilbert-base-uncased` if you want to compare.

In [ ]:
!python training/train_classifier.py \
    --dataset training/data/difficulty_dataset.jsonl \
    --output-dir training/checkpoints/difficulty-classifier \
    --epochs 3 \
    --batch-size 16

## Sanity-check the saved checkpoint

Loads it back and runs a couple of manual predictions, mirroring exactly what `app/router/classifier.py`'s `DifficultyClassifier.score()` does at request time -- if this cell's predictions look right, the router will behave the same way once you copy the checkpoint back into the repo.

In [ ]:
import json

import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

checkpoint_dir = "training/checkpoints/difficulty-classifier"

with open(f"{checkpoint_dir}/label_map.json") as f:
    label_map = json.load(f)

tokenizer = AutoTokenizer.from_pretrained(checkpoint_dir)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint_dir)
model.eval()

test_prompts = [
    "Hi, how are you?",
    "A train leaves station A at 60mph and another leaves station B at 40mph, 300 miles apart, heading toward each other. When do they meet, and show your work.",
    "What is the boiling point of water at sea level?",
]

for prompt in test_prompts:
    inputs = tokenizer(prompt, truncation=True, max_length=256, return_tensors="pt")
    with torch.no_grad():
        logits = model(**inputs).logits
        probs = torch.softmax(logits, dim=-1)[0]
        top_idx = int(torch.argmax(probs).item())
    print(f"{label_map[str(top_idx)]:>8} ({probs[top_idx]:.2f})  {prompt}")

## Download the checkpoint

Zips `training/checkpoints/difficulty-classifier/` and downloads it. Unzip on your machine into the same path (relative to the repo root, matching `classifier_checkpoint_dir` in `app/config.py`) -- the router picks it up automatically on the next request, no restart-specific code needed beyond the app already re-checking on first use.

In [ ]:
import shutil

from google.colab import files

shutil.make_archive("difficulty-classifier", "zip", "training/checkpoints/difficulty-classifier")
files.download("difficulty-classifier.zip")